# Метод опорных векторов (SVM): теория и практика

Этот ноутбук — теоретический и практический разбор алгоритма SVM. По аналогии с подходом, где используются синтетические данные для изолированной демонстрации эффектов, мы с нуля разберем максимизацию зазора, ядерный переход и оценку качества модели.

**Содержание:**
1. Геометрическая интуиция: гиперплоскости и максимизация зазора
2. Вариационное исчисление: Лагранжиан и условия Каруша-Куна-Таккера (KKT)
3. Двойственная задача и элементы функционального анализа (теорема Мерсера)
4. Реализация линейного SVM "с нуля" через градиентный спуск
5. Сравнение с использованием scikit-learn и оценка метрик

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)

## 1. Постановка задачи и максимизация зазора

Для обучающей выборки ищем разделяющую гиперплоскость $w^T x + b = 0$. Мы хотим максимизировать зазор (margin) между классами, который геометрически равен $\frac{2}{||w||}$. Максимизация $\frac{2}{||w||}$ эквивалентна минимизации $\frac{1}{2} ||w||^2$.

## 2. Вариационное исчисление: Лагранжиан и условия KKT

Задача условной оптимизации (Primal problem) для линейно разделимой выборки формулируется так:

$$\min_{w, b} \frac{1}{2} ||w||^2$$
при ограничениях: $y_i(w^T x_i + b) \ge 1 \quad \forall i$.

Используя методы вариационного исчисления, вводим множители Лагранжа $\alpha_i \ge 0$. Функция Лагранжа принимает вид:
$$\mathcal{L}(w, b, \alpha) = \frac{1}{2} ||w||^2 - \sum_{i=1}^n \alpha_i [y_i(w^T x_i + b) - 1]$$

Согласно условиям Каруша-Куна-Таккера (KKT), приравниваем к нулю производные по $w$ и $b$:
$$w = \sum_{i=1}^n \alpha_i y_i x_i \quad \text{и} \quad \sum_{i=1}^n \alpha_i y_i = 0$$

## 3. Двойственная задача и Функциональный анализ

Подстановка условий KKT обратно в Лагранжиан дает нам Двойственную задачу (Dual problem), которая зависит исключительно от скалярных произведений $\langle x_i, x_j \rangle$:

$$\max_{\alpha} \sum_{i=1}^n \alpha_i - \frac{1}{2} \sum_{i=1}^n \sum_{j=1}^n \alpha_i \alpha_j y_i y_j (x_i^T x_j)$$

Для линейно неразделимых данных применяется **Kernel Trick** (ядерный переход). В контексте функционального анализа и **теоремы Мерсера**, любая непрерывная, симметричная и положительно определенная функция $K(x, z)$ может быть представлена как скалярное произведение в некотором гильбертовом пространстве (RKHS). Это позволяет нам напрямую заменить $x_i^T x_j$ на функцию ядра $K(x_i, x_j)$, неявно перенося данные в бесконечномерное пространство для линейного разделения.

## 4. Реализация линейного SVM "с нуля"

На практике при проектировании пайплайнов алгоритм часто реализуют с использованием Hinge Loss функции и градиентного (или субградиентного) спуска. Ниже представлена реализация архитектуры базовой модели с нуля.

In [1]:
class SVM_from_scratch:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                # Условие удовлетворения Hinge Loss (margin >= 1)
                condition = y[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y[idx]))
                    self.b -= self.lr * y[idx]

    def predict(self, X):
        approx = np.dot(X, self.w) - self.b
        return np.sign(approx)

## 5. Сравнение со scikit-learn и оценка метрик

Используем сгенерированные синтетические данные для валидации нашей реализации и сравнения её с эталонной моделью `SVC` из библиотеки `scikit-learn`.

In [4]:
# Генерация линейно разделимых синтетических данных
X, y = make_classification(n_samples=300, n_features=2, n_informative=2, 
                           n_redundant=0, n_clusters_per_class=1, random_state=42)
# Для SVM метки классов должны быть -1 и 1
y = np.where(y == 0, -1, 1)

# Обучение реализации "с нуля"
model_scratch = SVM_from_scratch(n_iters=2000)
model_scratch.fit(X, y)

# Обучение scikit-learn модели
model_sklearn = SVC(kernel='linear', C=1.0)
model_sklearn.fit(X, y)

# Оценка метрик
print("Accuracy (модель с нуля):", accuracy_score(y, model_scratch.predict(X)))
print("Accuracy (scikit-learn):", accuracy_score(y, model_sklearn.predict(X)))

Accuracy (модель с нуля): 0.9233333333333333
Accuracy (scikit-learn): 0.9233333333333333
